# 06 · SECOP II: ¿cómo contrata la Alcaldía de Acacías?

**Proyecto:** ¿Acacías vive del petróleo? Regalías vs. desarrollo
**Fuente:** Colombia Compra Eficiente, *SECOP II – Contratos Electrónicos* (datos.gov.co), filtrado por Departamento = Meta y Ciudad contiene "Acac". Descarga del 24/11/2026, unas 9.900 filas.

**Preguntas**
1. ¿Cuánto contrata la alcaldía cada año y con qué modalidad?
2. ¿Qué peso tiene la **contratación directa**?
3. ¿Se nota la **Ley de Garantías** en el ritmo de contratación?
4. ¿Cuánto de lo contratado se paga con **regalías (SGR)**?
5. ¿Qué tan concentrados están los contratos en pocos proveedores?

⚠️ **Datos personales:** el archivo trae nombres y cédulas de personas naturales, cuentas bancarias, supervisores, etc. Es información pública, pero **no hace falta para el análisis**. Este notebook **descarta esas columnas** al cargar y **anonimiza** a los contratistas que son personas naturales. Solo se muestran por nombre las **empresas** (NIT).

In [ ]:
import hashlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option("display.float_format", "{:,.1f}".format)
pd.set_option("display.max_colwidth", 70)

RAW = Path("../data/secop contratos/SECOP_II_-_Contratos_Electrónicos_20261124.csv")
OUT = Path("../data processed")
ANIO_BASE = 2025

## 1. Cargar solo las columnas necesarias

In [ ]:
COLUMNAS = {
    "Nombre Entidad": "entidad", "Nit Entidad": "nit_entidad", "Orden": "orden", "Sector": "sector",
    "ID Contrato": "id_contrato", "Proceso de Compra": "proceso", "Estado Contrato": "estado",
    "Tipo de Contrato": "tipo_contrato", "Modalidad de Contratacion": "modalidad",
    "Justificacion Modalidad de Contratacion": "justificacion",
    "Fecha de Firma": "fecha_firma", "Fecha de Inicio del Contrato": "fecha_inicio", "Fecha de Fin del Contrato": "fecha_fin",
    "TipoDocProveedor": "tipo_doc_proveedor", "Documento Proveedor": "doc_proveedor", "Proveedor Adjudicado": "proveedor",
    "Es Pyme": "es_pyme", "Valor del Contrato": "valor", "Valor Pagado": "valor_pagado",
    "Sistema General de Regalías": "fuente_sgr", "Sistema General de Participaciones": "fuente_sgp",
    "Recursos Propios (Alcaldías y Gobernaciones)": "fuente_propios",
    "Presupuesto General de la Nacion – PGN": "fuente_pgn",
    "Objeto del Contrato": "objeto", "Dias adicionados": "dias_adicionados", "URLProceso": "url",
}

secop = pd.read_csv(RAW, dtype=str, usecols=list(COLUMNAS)).rename(columns=COLUMNAS)
print(secop.shape)
secop["entidad"].value_counts()

## 2. Limpieza

- **Valores** como texto con signo y puntos de miles: `$97.904.259`, a veces con coma decimal (`$97.904.258,59`).
- **Fechas** en formato estadounidense con AM/PM: `04/18/2022 03:00:57 PM`.
- **Estados** con mayúsculas inconsistentes (`terminado`, `Cerrado`).

In [ ]:
def a_numero(s):
    return pd.to_numeric(
        s.str.replace(r"[\$\s]", "", regex=True).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce")

for c in ["valor", "valor_pagado", "fuente_sgr", "fuente_sgp", "fuente_propios", "fuente_pgn", "dias_adicionados"]:
    secop[c] = a_numero(secop[c])

for c in ["fecha_firma", "fecha_inicio", "fecha_fin"]:
    secop[c] = pd.to_datetime(secop[c], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

secop["estado"] = secop["estado"].str.strip().str.capitalize()
secop["anio"] = secop["fecha_firma"].dt.year.astype("Int64")
secop["mes"] = secop["fecha_firma"].dt.to_period("M").dt.to_timestamp()

print("Contratos sin fecha de firma:", secop["fecha_firma"].isna().sum())
print("Firmas entre", secop["fecha_firma"].min().date(), "y", secop["fecha_firma"].max().date())
print("ID de contrato repetidos:", secop["id_contrato"].duplicated().sum())

### Agrupar modalidades

SECOP tiene muchas modalidades. Se agrupan en tres:
- **Directa:** sin competencia; la entidad escoge al contratista.
- **Régimen especial:** entidades con reglas propias (por ejemplo, el hospital, una ESE).
- **Competitiva:** hay convocatoria pública (mínima cuantía, selección abreviada, licitación, concurso de méritos).

In [ ]:
def agrupar_modalidad(m):
    m = str(m).lower()
    if "directa" in m:
        return "Directa"
    if "especial" in m:
        return "Régimen especial"
    return "Competitiva"

secop["modalidad_grupo"] = secop["modalidad"].map(agrupar_modalidad)
pd.crosstab(secop["modalidad"], secop["modalidad_grupo"])

### Anonimizar personas naturales

A cada persona natural se le asigna un código estable (por ejemplo `Persona natural 3f9a1c`) a partir de su documento. Así se puede contar cuántos contratos tiene cada contratista sin mostrar su nombre ni su cédula.

In [ ]:
es_persona = secop["tipo_doc_proveedor"].isin(["Cédula de Ciudadanía", "Cédula de Extranjería", "Tarjeta de Identidad",
                                               "Permiso por Protección Temporal"])
secop["tipo_proveedor"] = es_persona.map({True: "Persona natural", False: "Empresa / otro"})

def codigo(doc):
    return hashlib.sha256(str(doc).encode()).hexdigest()[:6]

secop["proveedor_publico"] = secop["proveedor"].where(~es_persona, "Persona natural " + secop["doc_proveedor"].map(codigo))
secop = secop.drop(columns=["proveedor", "doc_proveedor"])
secop["tipo_proveedor"].value_counts()

## 3. Foco: Alcaldía de Acacías

In [ ]:
ipc = pd.read_csv(OUT / "ipc_anual.csv").set_index("anio")[f"factor_a_pesos_{ANIO_BASE}"]

alc = secop[secop["entidad"] == "ALCALDIA DEL MUNICIPIO DE ACACIAS"].copy()
alc["valor_const"] = alc["valor"] * alc["anio"].map(ipc)
alc["sgr_const"] = alc["fuente_sgr"] * alc["anio"].map(ipc)

anual = alc.groupby("anio").agg(
    contratos=("id_contrato", "count"),
    valor_mm=("valor_const", lambda s: s.sum() / 1e9),
    sgr_mm=("sgr_const", lambda s: s.sum() / 1e9),
    pct_directa_contratos=("modalidad_grupo", lambda s: 100 * (s == "Directa").mean()),
    prestacion_servicios=("tipo_contrato", lambda s: (s == "Prestación de servicios").sum()),
)
anual["pct_valor_directa"] = 100 * alc[alc["modalidad_grupo"] == "Directa"].groupby("anio")["valor_const"].sum() / alc.groupby("anio")["valor_const"].sum()
anual

**Observaciones:**
- La alcaldía aparece en SECOP II **desde 2021**. Para los años anteriores haría falta descargar **SECOP I**.
- `valor_mm` y `sgr_mm` están en **miles de millones de pesos de 2025**.
- 2026 llega hasta noviembre.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

mod = (alc.groupby(["anio", "modalidad_grupo"])["valor_const"].sum().unstack().div(1e9))
mod.plot(kind="bar", stacked=True, ax=ax[0], width=0.8, color={"Directa": "#e07a5f", "Competitiva": "#2a6f97", "Régimen especial": "#9aa5b1"})
ax[0].set_title(f"Valor contratado por modalidad (miles de millones de pesos de {ANIO_BASE})")
ax[0].set_xlabel("")
ax[0].legend(frameon=False)

(alc.groupby("anio")["modalidad_grupo"].apply(lambda s: 100 * (s == "Directa").mean())
    .plot(kind="bar", ax=ax[1], color="#e07a5f", width=0.8))
ax[1].set_title("% de contratos firmados por contratación directa")
ax[1].set_xlabel("")
ax[1].set_ylim(0, 100)

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Lectura:** la gran mayoría de los **contratos** son directos, casi todos de **prestación de servicios** (personas contratadas para apoyar a la alcaldía). Pero en **valor** pesan más los procesos competitivos, porque las obras grandes sí se licitan.

## 4. Ley de Garantías

La Ley 996 de 2005 **prohíbe la contratación directa** durante los cuatro meses antes de la elección presidencial y hasta la segunda vuelta:
- **2022:** del 29 de enero al 19 de junio.
- **2026:** del 31 de enero al 21 de junio.

Si las entidades se apuran a contratar antes de la restricción, debería verse un **pico en enero** y un **vacío** durante la prohibición.

In [ ]:
mensual = alc.groupby(["mes", "modalidad_grupo"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(mensual.index, mensual.get("Directa", 0), width=25, color="#e07a5f", label="Directa")
ax.bar(mensual.index, mensual.drop(columns="Directa", errors="ignore").sum(axis=1),
       bottom=mensual.get("Directa", 0), width=25, color="#2a6f97", label="Otras modalidades")
for ini, fin in [("2022-01-29", "2022-06-19"), ("2026-01-31", "2026-06-21")]:
    ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fin), color="#f2cc8f", alpha=0.5)
ax.text(pd.Timestamp("2022-02-05"), ax.get_ylim()[1] * 0.9, "Ley de Garantías", fontsize=9)
ax.text(pd.Timestamp("2026-02-05"), ax.get_ylim()[1] * 0.9, "Ley de Garantías", fontsize=9)
ax.set_title("Alcaldía de Acacías: contratos firmados por mes")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
ene_26 = (alc["mes"] == "2026-01-01").sum()
garantias_26 = alc["fecha_firma"].between("2026-01-31", "2026-06-21").sum()
prom_mes_25 = len(alc[alc["anio"] == 2025]) / 12
print(f"Enero 2026: {ene_26} contratos (promedio mensual 2025: {prom_mes_25:.0f})")
print(f"31 ene - 21 jun 2026 (Ley de Garantías): {garantias_26} contratos en casi 5 meses")

**Lectura:** cada año arranca con un pico de contratos en enero y febrero (en 2024 fue en febrero, por el cambio de gobierno municipal). En los años normales ese arranque suma entre 226 y 353 contratos. En 2026, como la Ley de Garantías prohibía la contratación directa desde el 31 de enero, todo se adelantó: **618 contratos solo en enero**, y durante la restricción la contratación casi se detuvo. Es un comportamiento **legal** y muy común en todo el país, pero muestra cómo el calendario electoral mueve la contratación.

## 5. Contratos pagados con regalías

In [ ]:
sgr = alc[alc["fuente_sgr"] > 0].copy()
print(f"Contratos con recursos SGR: {len(sgr)} de {len(alc)}")
print(f"Valor SGR: {sgr['sgr_const'].sum() / 1e9:,.1f} mil millones de pesos de {ANIO_BASE}")
print()
print(sgr.groupby("modalidad_grupo")["sgr_const"].sum().div(1e9).round(1).rename("SGR mm"))

(sgr.nlargest(10, "fuente_sgr")
    [["fecha_firma", "tipo_contrato", "modalidad", "proveedor_publico", "fuente_sgr", "objeto"]]
    .assign(fuente_sgr=lambda d: d["fuente_sgr"] / 1e9, objeto=lambda d: d["objeto"].str[:90])
    .rename(columns={"fuente_sgr": "sgr_mm_corrientes"}))

Los contratos grandes con regalías son **obras, interventorías y suministros**, la mayoría por procesos competitivos. Se pueden cruzar con los proyectos del notebook 04 buscando el nombre de la obra en el objeto del contrato.

### Cruce con los proyectos "para seguimiento" del notebook 04

En MapaInversiones, el puente sobre el caño Chichimene y la vivienda Balcones del Camino Real aparecían con **0% de avance físico** y el plazo vencido. ¿Qué dice SECOP de sus contratos?

In [ ]:
claves = "CHICHIMENE EN LA VEREDA QUEBRADITAS|BALCONES DEL CAMINO REAL|ALCANTARILLADO PLUVIAL SOBRE LA CARRERA 23"
cruce = alc[alc["objeto"].str.contains(claves, case=False, na=False) & (alc["tipo_proveedor"] == "Empresa / otro")]
cruce[["fecha_firma", "tipo_contrato", "estado", "valor", "valor_pagado", "fecha_fin", "dias_adicionados", "proveedor_publico"]]     .assign(valor=lambda d: d["valor"] / 1e9, valor_pagado=lambda d: d["valor_pagado"] / 1e9)     .rename(columns={"valor": "valor_mm", "valor_pagado": "pagado_mm"})

**Lectura:** los contratos existen, fueron **licitados**, están **modificados con prórrogas** (cientos de días adicionados) y en buena parte **pagados**. Eso es consistente con obras que avanzaron pero se demoraron, y refuerza que el 0% físico de MapaInversiones es más un **problema de reporte** que una obra fantasma. Para confirmarlo del todo, se revisan los informes de interventoría en el enlace del proceso (columna `url`).

## 6. Concentración de proveedores

In [ ]:
por_proveedor = (alc.groupby(["proveedor_publico", "tipo_proveedor"])
                    .agg(contratos=("id_contrato", "count"), valor_mm=("valor_const", lambda s: s.sum() / 1e9))
                    .sort_values("valor_mm", ascending=False)
                    .reset_index())

total = por_proveedor["valor_mm"].sum()
top10 = por_proveedor.head(10).copy()
top10["pct_del_total"] = 100 * top10["valor_mm"] / total
print(f"{len(por_proveedor):,} proveedores distintos")
print(f"Los 10 mayores concentran el {top10['pct_del_total'].sum():.1f}% del valor contratado (2021-2026)")
top10.index += 1
top10

In [ ]:
# ¿Cuántas personas tienen contratos de prestación de servicios cada año?
ops = alc[(alc["tipo_contrato"] == "Prestación de servicios") & (alc["tipo_proveedor"] == "Persona natural")]
ops.groupby("anio").agg(
    personas=("proveedor_publico", "nunique"),
    contratos=("id_contrato", "count"),
    valor_mediano_millones=("valor", lambda s: s.median() / 1e6),
)

**Lectura:** cada año la alcaldía contrata **cientos de personas por prestación de servicios**, más que su planta de empleados en muchos municipios. Es un patrón común en Colombia, a menudo llamado "nómina paralela".

## 7. Exportar

In [ ]:
secop.to_csv(OUT / "secop2_acacias_limpio_anonimizado.csv", index=False, encoding="utf-8-sig")
anual.to_csv(OUT / "secop_alcaldia_anual.csv", encoding="utf-8-sig")
por_proveedor.to_csv(OUT / "secop_alcaldia_proveedores.csv", index=False, encoding="utf-8-sig")
mensual.to_csv(OUT / "secop_alcaldia_mensual.csv", encoding="utf-8-sig")
print("Listo:", sorted(p.name for p in OUT.glob("secop*.csv")))

## Notas para el análisis

- **Contratación directa no es ilegal.** La ley la permite para servicios profesionales y de apoyo a la gestión, entre otros casos. Lo interesante es su **peso** y su relación con el calendario electoral.
- **SECOP II arranca en 2021 para la alcaldía.** Para ver 2013-2020 hace falta SECOP I.
- **Valor del contrato ≠ valor pagado:** los contratos pueden modificarse, adicionarse o no ejecutarse completos.
- En el portafolio **no publiques nombres ni cédulas** de personas naturales; usa los códigos anónimos.